# 01b – Explorative Datenanalyse: Berliner U-Bahn

Dieses Notebook untersucht die gesammelten Echtzeit-Abfahrtsdaten der Berliner U-Bahn.
Voraussetzung: Elasticsearch läuft (`docker-compose up -d`) und der Collector hat mindestens einige Stunden Daten gesammelt.

## 0 – Setup

In [ ]:
import warnings, sys, pathlib
warnings.filterwarnings("ignore")

# Add project root to path so config.settings is importable from notebooks/
_root = pathlib.Path.cwd()
while not (_root / "config").exists() and _root != _root.parent:
    _root = _root.parent
sys.path.insert(0, str(_root))

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import folium
from scipy import stats
from elasticsearch import Elasticsearch
from elasticsearch.helpers import scan
from config.settings import ES_HOST, ES_USER, ES_PASSWORD

# Elasticsearch-Verbindung
es = Elasticsearch(ES_HOST, basic_auth=(ES_USER, ES_PASSWORD))
INDEX = "ubahn-departures"

# Verbindungscheck
assert es.ping(), "Elasticsearch nicht erreichbar – docker-compose up -d?"
count = es.count(index=INDEX)["count"]
print(f"Verbindung OK – {count:,} Dokumente in '{INDEX}'")

## 1 – Datenübersicht

Alle Dokumente aus dem Index `ubahn-departures` werden via Scroll-API geladen und in einen DataFrame überführt.

In [ ]:
# Zufalls-Sample laden (~0.3% der Dokumente via random_score)
SAMPLE_RATE = 0.003  # ~93k aus 31M Docs

hits = scan(
    es,
    index=INDEX,
    query={
        "query": {
            "function_score": {
                "query": {"match_all": {}},
                "functions": [{"random_score": {"seed": 42, "field": "_seq_no"}}],
                "boost_mode": "replace",
            }
        },
        "min_score": 1 - SAMPLE_RATE,
    },
    size=5000,
)

records = [hit["_source"] for hit in hits]
df = pd.DataFrame(records)

# Datumstypen konvertieren
for col in ["planned_when", "when", "collected_at"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], utc=True)

print(f"Geladene Zeilen: {len(df):,} (Sample ~{SAMPLE_RATE*100:.1f}% des Index)")
df.info()

In [ ]:
df.describe(include="all").T

In [ ]:
print("Zeitraum (planned_when):")
print(f"  Von:  {df['planned_when'].min()}")
print(f"  Bis:  {df['planned_when'].max()}")
print(f"  Dauer: {(df['planned_when'].max() - df['planned_when'].min()).days} Tage")
print()
print(f"Unique Linien ({df['line_name'].nunique()}): {sorted(df['line_name'].dropna().unique())}")
print(f"Unique Haltestellen: {df['stop_name'].nunique()}")

## 2 – Datenqualität

Fehlende `delay_s`-Werte entstehen, wenn die BVG-API für eine Abfahrt keine Echtzeit-Information liefert.
Hohe Fehlquoten deuten auf Linien oder Haltestellen mit schlechter Echtzeit-Abdeckung hin.

In [ ]:
# Anteil fehlender delay_s pro Linie
missing_by_line = (
    df.groupby("line_name")["delay_s"]
    .apply(lambda s: s.isna().mean() * 100)
    .reset_index()
    .rename(columns={"delay_s": "fehlend_pct"})
    .sort_values("fehlend_pct", ascending=False)
)

fig = px.bar(
    missing_by_line,
    x="line_name",
    y="fehlend_pct",
    title="Anteil fehlender Verspätungswerte pro Linie (%)",
    labels={"line_name": "Linie", "fehlend_pct": "Fehlend (%)"},
    color="fehlend_pct",
    color_continuous_scale="Reds",
)
fig.update_layout(coloraxis_showscale=False)
fig.show()

In [ ]:
# Anteil fehlender delay_s pro Haltestelle (Top 20)
missing_by_stop = (
    df.groupby("stop_name")["delay_s"]
    .apply(lambda s: s.isna().mean() * 100)
    .reset_index()
    .rename(columns={"delay_s": "fehlend_pct"})
    .sort_values("fehlend_pct", ascending=False)
    .head(20)
)
missing_by_stop.index = range(1, len(missing_by_stop) + 1)
missing_by_stop.columns = ["Haltestelle", "Fehlend (%)"]
missing_by_stop["Fehlend (%)"] = missing_by_stop["Fehlend (%)"].round(1)
missing_by_stop

In [ ]:
# Heatmap: Datenverfügbarkeit (Stunde × Wochentag)
TAGE = ["Mo", "Di", "Mi", "Do", "Fr", "Sa", "So"]

avail = (
    df.groupby(["day_of_week", "hour_of_day"])["delay_s"]
    .apply(lambda s: (1 - s.isna().mean()) * 100)
    .reset_index()
    .rename(columns={"delay_s": "verfuegbar_pct"})
    .pivot(index="day_of_week", columns="hour_of_day", values="verfuegbar_pct")
    .reindex(index=range(7), columns=range(24))
)

fig = px.imshow(
    avail,
    labels={"x": "Stunde", "y": "Wochentag", "color": "Verfügbar (%)"},
    y=TAGE,
    title="Datenverfügbarkeit: Echtzeit-Verspätung (Stunde × Wochentag)",
    color_continuous_scale="RdYlGn",
    zmin=0, zmax=100,
    aspect="auto",
)
fig.show()

## 3 – Verspätungsverteilung

Für alle weiteren Analysen werden nur Zeilen mit vorhandenem `delay_s` verwendet.
Negative Werte bedeuten Verfrühung, positive Werte Verspätung.

In [ ]:
df_valid = df.dropna(subset=["delay_s"]).copy()
df_valid["delay_min"] = df_valid["delay_s"] / 60

print(f"Zeilen mit delay_s: {len(df_valid):,} ({len(df_valid)/len(df)*100:.1f}%)")
print(df_valid["delay_s"].describe().round(1))

In [ ]:
# Histogramm: alle Werte und gekappt auf ±600s
from plotly.subplots import make_subplots

fig = make_subplots(rows=1, cols=2,
    subplot_titles=["Alle Werte", "Ohne Ausreißer (±500 s)"])

fig.add_trace(go.Histogram(
    x=df_valid["delay_s"], nbinsx=100,
    marker_color="steelblue", name="Alle"
), row=1, col=1)

df_clip = df_valid[df_valid["delay_s"].between(-500, 500)]
fig.add_trace(go.Histogram(
    x=df_clip["delay_s"], nbinsx=80,
    marker_color="cornflowerblue", name="±600 s"
), row=1, col=2)

fig.update_xaxes(title_text="Verspätung (s)", row=1, col=1)
fig.update_xaxes(title_text="Verspätung (s)", row=1, col=2)
fig.update_yaxes(title_text="Anzahl Abfahrten", row=1, col=1)
fig.update_layout(title="Histogramm der Verspätungen", showlegend=False)
fig.show()

In [ ]:
# Boxplot: Verspätung pro Linie
line_order = (
    df_clip.groupby("line_name")["delay_s"]
    .median()
    .sort_values(ascending=False)
    .index.tolist()
)

fig = px.box(
    df_clip,
    x="line_name",
    y="delay_s",
    category_orders={"line_name": line_order},
    title="Verspätungsverteilung pro Linie (±600 s)",
    labels={"line_name": "Linie", "delay_s": "Verspätung (s)"},
    color="line_name",
)
fig.update_layout(showlegend=False)
fig.add_hline(y=0, line_dash="dash", line_color="black", opacity=0.4)
fig.show()

In [ ]:
# Tageszeit-Kategorien
def tageszeit(hour: int) -> str:
    if 6 <= hour < 9 or 16 <= hour < 19:
        return "Rush (6-9 / 16-19 Uhr)"
    elif 9 <= hour < 16 or 19 <= hour < 22:
        return "Off-Peak (9-16 / 19-22 Uhr)"
    else:
        return "Nacht (22-6 Uhr)"

df_clip = df_clip.copy()
df_clip["tageszeit"] = df_clip["hour_of_day"].apply(tageszeit)

fig = px.violin(
    df_clip,
    x="tageszeit",
    y="delay_s",
    box=True,
    color="tageszeit",
    category_orders={"tageszeit": ["Rush (6-9 / 16-19 Uhr)", "Off-Peak (9-16 / 19-22 Uhr)", "Nacht (22-6 Uhr)"]},
    title="Verspätungsverteilung nach Tageszeit",
    labels={"tageszeit": "Tageszeit", "delay_s": "Verspätung (s)"},
)
fig.update_layout(showlegend=False)
fig.add_hline(y=0, line_dash="dash", line_color="black", opacity=0.4)
fig.show()

## 4 – Zeitliche Muster

Wie entwickelt sich die mittlere Verspätung über den Tag und die Woche?

In [ ]:
# Mittlere Verspätung pro Stunde
by_hour = (
    df_clip.groupby("hour_of_day")["delay_s"]
    .agg(mean="mean", sem=lambda s: s.sem())
    .reset_index()
)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=by_hour["hour_of_day"],
    y=by_hour["mean"],
    error_y=dict(type="data", array=by_hour["sem"] * 1.96, visible=True),
    mode="lines+markers",
    line=dict(color="steelblue", width=2),
    name="Mittlere Verspätung",
))
fig.add_hline(y=0, line_dash="dash", line_color="black", opacity=0.4)
fig.update_layout(
    title="Mittlere Verspätung nach Tagesstunde (95%-KI)",
    xaxis_title="Stunde",
    yaxis_title="Mittlere Verspätung (s)",
    xaxis=dict(tickmode="linear", dtick=1),
)
fig.show()

In [ ]:
# Heatmap: mittlere Verspätung (Wochentag × Stunde)
heat = (
    df_clip.groupby(["day_of_week", "hour_of_day"])["delay_s"]
    .mean()
    .reset_index()
    .pivot(index="day_of_week", columns="hour_of_day", values="delay_s")
    .reindex(index=range(7), columns=range(24))
)

fig = px.imshow(
    heat,
    labels={"x": "Stunde", "y": "Wochentag", "color": "Ø Verspätung (s)"},
    y=TAGE,
    title="Mittlere Verspätung: Wochentag × Stunde (s)",
    color_continuous_scale="RdYlGn_r",
    aspect="auto",
)
fig.show()

In [ ]:
# Top-3 Linien nach mittlerer Verspätung
top3_lines = (
    df_clip.groupby("line_name")["delay_s"]
    .mean()
    .nlargest(3)
    .index.tolist()
)

by_hour_line = (
    df_clip[df_clip["line_name"].isin(top3_lines)]
    .groupby(["line_name", "hour_of_day"])["delay_s"]
    .mean()
    .reset_index()
)

fig = px.line(
    by_hour_line,
    x="hour_of_day",
    y="delay_s",
    color="line_name",
    markers=True,
    title="Tagesverlauf der mittleren Verspätung: Top-3 Linien",
    labels={"hour_of_day": "Stunde", "delay_s": "Mittlere Verspätung (s)", "line_name": "Linie"},
)
fig.add_hline(y=0, line_dash="dash", line_color="black", opacity=0.4)
fig.update_layout(xaxis=dict(tickmode="linear", dtick=1))
fig.show()

## 5 – Geo-Analyse

Jede Haltestelle wird als Kreis auf einer Karte dargestellt.
Radius und Farbe kodieren die mittlere Verspätung an der jeweiligen Haltestelle.

In [ ]:
# Mittlere Verspätung pro Haltestelle inkl. Koordinaten
def extract_lat(loc):
    if isinstance(loc, dict):
        return loc.get("lat")
    return None

def extract_lon(loc):
    if isinstance(loc, dict):
        return loc.get("lon")
    return None

df_clip["lat"] = df_clip["stop_location"].apply(extract_lat)
df_clip["lon"] = df_clip["stop_location"].apply(extract_lon)

stop_stats = (
    df_clip.dropna(subset=["lat", "lon"])
    .groupby("stop_name")
    .agg(
        lat=("lat", "first"),
        lon=("lon", "first"),
        mean_delay=("delay_s", "mean"),
        n=("delay_s", "count"),
        lines=("line_name", lambda s: ", ".join(sorted(s.dropna().unique()))),
    )
    .reset_index()
)

print(f"{len(stop_stats)} Haltestellen mit Geo-Koordinaten und Verspätungsdaten")

In [ ]:
# Folium-Karte
def delay_color(delay_s: float) -> str:
    if delay_s < 60:
        return "green"
    elif delay_s < 120:
        return "orange"
    else:
        return "red"

center_lat = stop_stats["lat"].mean()
center_lon = stop_stats["lon"].mean()

m = folium.Map(location=[center_lat, center_lon], zoom_start=12, tiles="CartoDB positron")

delay_max = stop_stats["mean_delay"].quantile(0.95)

for _, row in stop_stats.iterrows():
    radius = 4 + (row["mean_delay"] / delay_max) * 12
    radius = max(4, min(radius, 18))
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=radius,
        color=delay_color(row["mean_delay"]),
        fill=True,
        fill_color=delay_color(row["mean_delay"]),
        fill_opacity=0.7,
        popup=folium.Popup(
            f"<b>{row['stop_name']}</b><br>"
            f"Linien: {row['lines']}<br>"
            f"Ø Verspätung: {row['mean_delay']:.0f} s ({row['mean_delay']/60:.1f} min)<br>"
            f"Abfahrten: {row['n']:,}",
            max_width=250,
        ),
        tooltip=f"{row['stop_name']}: {row['mean_delay']:.0f} s",
    ).add_to(m)

# Legende
legend_html = """
<div style="position: fixed; bottom: 30px; left: 30px; z-index: 1000;
            background: white; padding: 10px; border-radius: 6px;
            border: 1px solid #ccc; font-size: 13px; line-height: 1.8;">
    <b>Ø Verspätung</b><br>
    <span style='color:green'>&#9679;</span> &lt; 60 s<br>
    <span style='color:orange'>&#9679;</span> 60–120 s<br>
    <span style='color:red'>&#9679;</span> &gt; 120 s
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))
m

## 6 – Top-Hotspots

Welche Haltestellen haben die höchste mittlere Verspätung, und wie verteilt sich diese über den Tag?

In [ ]:
# Top-10 Haltestellen nach mittlerer Verspätung
top10 = (
    stop_stats
    .nlargest(10, "mean_delay")
    .reset_index(drop=True)
)
top10.index = range(1, 11)

top10_display = top10[["stop_name", "lines", "mean_delay", "n"]].copy()
top10_display.columns = ["Haltestelle", "Linien", "Ø Verspätung (s)", "Abfahrten"]
top10_display["Ø Verspätung (s)"] = top10_display["Ø Verspätung (s)"].round(1)
top10_display["Ø Verspätung (min)"] = (top10_display["Ø Verspätung (s)"] / 60).round(2)
top10_display

In [ ]:
# Verspätungsverlauf über den Tag für Top-3 Haltestellen
top3_stops = top10["stop_name"].head(3).tolist()

hotspot_hourly = (
    df_clip[df_clip["stop_name"].isin(top3_stops)]
    .groupby(["stop_name", "hour_of_day"])["delay_s"]
    .mean()
    .reset_index()
)

fig = px.line(
    hotspot_hourly,
    x="hour_of_day",
    y="delay_s",
    color="stop_name",
    markers=True,
    title="Tagesverlauf der mittleren Verspätung: Top-3 Hotspot-Haltestellen",
    labels={
        "hour_of_day": "Stunde",
        "delay_s": "Mittlere Verspätung (s)",
        "stop_name": "Haltestelle",
    },
)
fig.add_hline(y=0, line_dash="dash", line_color="black", opacity=0.4)
fig.update_layout(xaxis=dict(tickmode="linear", dtick=1))
fig.show()

In [ ]:
# Balkendiagramm Top-10 zur Übersicht
fig = px.bar(
    top10.sort_values("mean_delay"),
    x="mean_delay",
    y="stop_name",
    orientation="h",
    color="mean_delay",
    color_continuous_scale="Reds",
    title="Top-10 Haltestellen nach mittlerer Verspätung",
    labels={"stop_name": "Haltestelle", "mean_delay": "Ø Verspätung (s)"},
    text=top10.sort_values("mean_delay")["mean_delay"].round(0).astype(int).astype(str) + " s",
)
fig.update_traces(textposition="outside")
fig.update_layout(coloraxis_showscale=False, yaxis_title="")
fig.show()